# 17.10 壓縮後怎麼實際運算？

# 第 17 章：量化，用更少位元表示模型

前置：第2章Linear與數值。像把連續刻度尺改成較粗的格子：先看誤差，再看真正的儲存bytes，最後才談硬體加速。quantization.py的參考Linear會反量化，不會假裝是低位元kernel。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：換較粗刻度尺：省格子但有誤差**

縮小儲存與硬體加速分開驗證。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/quantization.svg")))

**先想一想：**buffer變小就可宣傳decode加速嗎？

壓得小與算得快需要不同證據。專門kernel直接利用低位元weight；參考反量化實現可能較慢甚至臨時分配大浮點矩陣。

**把直覺寫成數學：**記錄文件大小、載入峯值、prefill/decode、execution peak；不要只看numel。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.quantization import QuantizedLinear

layer = nn.Linear(64, 32)
compressed = QuantizedLinear(layer, 4)
float_bytes = sum(p.numel() * p.element_size() for p in layer.parameters())
print("權重payload", float_bytes, "compressed buffers", compressed.storage_bytes())
print("低位元backend需選實際硬體支持；本參考版不聲稱kernel加速")

**如何讀結果：**TorchAO等是可選硬體延伸，不增加隱藏依賴；接口和支持列表會隨版本變化。

**只改一件事：**只增加每channel scale overhead，覈對儲存收益。
